# Graph2Edits 教程 2：数据处理

## 概述

本 notebook 用最小化、可读的代码，把 Graph2Edits 的数据处理主线完整拆开：

`原始反应 -> canonicalized reaction -> edit 序列 -> vocab -> 逐步图状态 -> batch 张量`

重点不是复现整个工程的批量脚本，而是把 `preprocess.py` 和 `prepare_data.py` 里的核心逻辑改写成适合教学观察的小样本流程。

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 载入 demo 反应并回顾 raw / canonicalized 的关系 |
| 2 | 生成每条反应的 edit 序列 |
| 3 | 按训练集逻辑构造 vocab |
| 4 | 把 edit 序列展开成逐步图状态 |
| 5 | 生成 batch 张量并保存教学版产物 |
| 6 | 对照原仓库的数据文件流转 |

> 提示
>
> 这里使用的 demo 数据只保留 3 条反应，但保留了 `Change Atom`、`Delete Bond`、`Change Bond`、`Attaching LG` 这几类最关键的 edit 类型。

## 源码对应

- 文件：`source_repos/Graph2Edits/canonicalize_prod.py`
- 文件：`source_repos/Graph2Edits/preprocess.py`
- 文件：`source_repos/Graph2Edits/prepare_data.py`
- 文件：`source_repos/Graph2Edits/utils/generate_edits.py`
- 文件：`source_repos/Graph2Edits/utils/collate_fn.py`

In [1]:
from pathlib import Path
import copy
import sys
from collections import Counter

import joblib
import pandas as pd
import torch
from rdkit import Chem


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "source_repos").exists() and (candidate / "teaching_demos").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录，请从 Chemical_Synthesis 项目内启动 notebook。")


PROJECT_ROOT = find_project_root(Path.cwd())
TUTORIAL_REL = Path("teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits")
REPO_REL = Path("source_repos/Graph2Edits")
TUTORIAL_ROOT = PROJECT_ROOT / TUTORIAL_REL
REPO_ROOT = PROJECT_ROOT / REPO_REL
DEMO_DATA_FILE = TUTORIAL_ROOT / "data/demo_reactions.csv"
PROCESSED_TRAIN_ROOT = TUTORIAL_ROOT / "data/processed/uspto_50k_demo/train"
TENSOR_ROOT = PROCESSED_TRAIN_ROOT / "without_rxn_class"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from canonicalize_prod import remap_rxn_smi
from prepare_data import apply_edit_to_mol
from utils.collate_fn import get_batch_graphs, prepare_edit_labels
from utils.datasets import RetroEditDataset, RetroEvalDataset
from utils.generate_edits import generate_reaction_edits
from utils.reaction_actions import Termination
from utils.rxn_graphs import MolGraph, RxnGraph, Vocab


def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


pd.set_option("display.max_colwidth", 120)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEMO_DATA_FILE:", DEMO_DATA_FILE)
print("PROCESSED_TRAIN_ROOT:", PROCESSED_TRAIN_ROOT)

PROJECT_ROOT: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
DEMO_DATA_FILE: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/demo_reactions.csv
PROCESSED_TRAIN_ROOT: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train


## Step 1: 从 raw reaction 到 canonicalized reaction

### 为什么需要这一步？

Graph2Edits 在 `USPTO_50k` 上并不是直接读取 `raw_train.csv`，而是先通过 `canonicalize_prod.py` 把产物 canonicalize，并重新整理 atom map。这样后续生成 edit 序列时，反应物和产物的原子对应关系更稳定。

### 核心思想

- `raw_reaction`：来自原始 CSV。
- `canonicalized_reaction`：经过 `remap_rxn_smi()` 标准化后的版本。
- 教学版只演示 1 条反应的转换，但保留原始 3 条 canonicalized 示例用于后续处理。

### 源码对应

- 文件：`source_repos/Graph2Edits/canonicalize_prod.py`
- 函数：`canonicalize_prod()`
- 函数：`infer_correspondence()`
- 函数：`remap_rxn_smi()`

In [2]:
demo_df = pd.read_csv(DEMO_DATA_FILE)
display(demo_df[["source_index", "id", "class", "expected_edit_pattern"]])

example = demo_df.iloc[1]
canonicalized_rxn, correspondence = remap_rxn_smi(example["raw_reaction"])

print("example id:", example["id"])
print("raw reaction:")
print(example["raw_reaction"])
print("\ncanonicalized reaction from demo file:")
print(example["canonicalized_reaction"])
print("\ncanonicalized reaction regenerated by remap_rxn_smi:")
print(canonicalized_rxn)
print("\natom-map correspondence preview:")
print(list(correspondence.items())[:10])

assert canonicalized_rxn == example["canonicalized_reaction"]

,source_index,id,class,expected_edit_pattern
0,0,US05849732,6,Change Atom + Attaching LG
1,1,US20120114765A1,2,Delete Bond + Attaching LG
2,7,US20130005716A1,7,Change Bond


example id: US20120114765A1
raw reaction:
O[C:1](=[O:2])[c:3]1[cH:4][c:5]([N+:6](=[O:7])[O-:8])[c:9]([S:10][c:11]2[c:12]([Cl:13])[cH:14][n:15][cH:16][c:17]2[Cl:18])[s:19]1.[NH2:20][c:21]1[cH:22][cH:23][cH:24][c:25]2[cH:26][n:27][cH:28][cH:29][c:30]12>>[C:1](=[O:2])([c:3]1[cH:4][c:5]([N+:6](=[O:7])[O-:8])[c:9]([S:10][c:11]2[c:12]([Cl:13])[cH:14][n:15][cH:16][c:17]2[Cl:18])[s:19]1)[NH:20][c:21]1[cH:22][cH:23][cH:24][c:25]2[cH:26][n:27][cH:28][cH:29][c:30]12

canonicalized reaction from demo file:
[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2]([c:14]1[cH:15][c:16]([N+:17](=[O:18])[O-:19])[c:20]([S:21][c:22]2[c:23]([Cl:24])[cH:25][n:26][cH:27][c:28]2[Cl:29])[s:30]1)[OH:31]>>[O:1]=[C:2]([NH:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12)[c:14]1[cH:15][c:16]([N+:17](=[O:18])[O-:19])[c:20]([S:21][c:22]2[c:23]([Cl:24])[cH:25][n:26][cH:27][c:28]2[Cl:29])[s:30]1

canonicalized reaction regenerated by remap_rxn_smi:
[NH2:3][c:4]1[cH:5][cH:6

## Step 2: 生成 edit 序列

### 为什么需要这一步？

Graph2Edits 的训练目标不是直接输出 reactants，而是预测“当前产物图下一步应该做什么 edit”。因此每条反应都要先被翻译成一个 edit 序列。

### 核心思想

`generate_reaction_edits()` 会比较反应物和产物，依次抽取：

- `Delete Bond`
- `Change Bond`
- `Change Atom`
- `Attaching LG`
- `Terminate`

同时还会记录每一步 edit 对应的原子位置 `edits_atom`。

### 源码对应

- 文件：`source_repos/Graph2Edits/utils/generate_edits.py`
- 函数：`generate_reaction_edits()`
- 依赖：`utils/chem.py` 里的 `get_atom_info()`、`get_bond_info()`、`align_kekulize_pairs()`

In [3]:
reaction_objects = []
edit_rows = []

for _, row in demo_df.iterrows():
    rxn_data = generate_reaction_edits(
        row["canonicalized_reaction"],
        kekulize=True,
        rxn_class=int(row["class"]) - 1,
        rxn_id=row["id"],
    )
    reaction_objects.append(rxn_data)

    for step_idx, edit in enumerate(rxn_data.edits):
        if edit == "Terminate":
            edit_rows.append(
                {
                    "id": row["id"],
                    "step": step_idx,
                    "edit_type": "Terminate",
                    "edit_payload": "-",
                    "edit_atom": "-",
                }
            )
        else:
            edit_rows.append(
                {
                    "id": row["id"],
                    "step": step_idx,
                    "edit_type": edit[0],
                    "edit_payload": repr(edit[1]),
                    "edit_atom": repr(rxn_data.edits_atom[step_idx]),
                }
            )

edit_df = pd.DataFrame(edit_rows)
display(edit_df)
print("step count distribution:", Counter(len(r.edits) for r in reaction_objects))

,id,step,edit_type,edit_payload,edit_atom
0,US05849732,0,Change Atom,"(1, 0)",10
1,US05849732,1,Attaching LG,'*C(=O)OCc1ccccc1',10
2,US05849732,2,Terminate,-,-
3,US20120114765A1,0,Delete Bond,"(None, None)","[2, 3]"
4,US20120114765A1,1,Attaching LG,'*O',2
5,US20120114765A1,2,Terminate,-,-
6,US20130005716A1,0,Change Bond,"(2, 0)","[7, 8]"
7,US20130005716A1,1,Terminate,-,-


step count distribution: Counter({3: 2, 2: 1})


## Step 3: 按训练集逻辑构造 vocab

### 为什么需要这一步？

模型最后输出的不是自由文本，而是一个离散 edit 分类空间。因此训练前必须先统计训练集里出现过的 edit，并把它们整理成 `bond_vocab`、`atom_vocab`、`lg_vocab`、`atom_lg_vocab`。

### 核心思想

这里沿用 `preprocess.py` 在 `uspto_50k` 训练集上的规则：

- `Delete Bond` / `Change Bond` 进入 `bond_vocab`
- `Change Atom` 进入 `atom_vocab`
- `Attaching LG` 进入 `lg_vocab`
- `atom_lg_vocab = atom_vocab + lg_vocab`

教学版不做低频 leaving group 过滤，因为样本本来就只有 3 条。

### 源码对应

- 文件：`source_repos/Graph2Edits/preprocess.py`
- 函数：`preprocessing()`
- 说明：教学版保留词表构造主线，但省略大规模过滤与磁盘批处理。

In [4]:
all_edits = Counter()

for rxn_data in reaction_objects:
    for edit in rxn_data.edits:
        if edit != "Terminate":
            all_edits[edit] += 1

atom_edits = []
bond_edits = []
lg_edits = []
atom_lg_edits = []

for edit, count in all_edits.items():
    if edit[0] == "Change Atom":
        atom_edits.append(edit)
        atom_lg_edits.append(edit)
    elif edit[0] in {"Delete Bond", "Change Bond", "Add Bond"}:
        bond_edits.append(edit)
    elif edit[0] == "Attaching LG":
        lg_edits.append(edit)

atom_lg_edits.extend(lg_edits)

bond_vocab = Vocab(bond_edits)
atom_vocab = Vocab(atom_lg_edits)

vocab_summary = pd.DataFrame(
    [
        {"vocab": "bond_vocab", "size": len(bond_vocab), "elements": [repr(x) for x in bond_edits]},
        {"vocab": "atom_vocab", "size": len(Vocab(atom_edits)), "elements": [repr(x) for x in atom_edits]},
        {"vocab": "lg_vocab", "size": len(Vocab(lg_edits)), "elements": [repr(x) for x in lg_edits]},
        {"vocab": "atom_lg_vocab", "size": len(atom_vocab), "elements": [repr(x) for x in atom_lg_edits]},
    ]
)

display(vocab_summary)

,vocab,size,elements
0,bond_vocab,2,"[('Delete Bond', (None, None)), ('Change Bond', (2, 0))]"
1,atom_vocab,1,"[('Change Atom', (1, 0))]"
2,lg_vocab,2,"[('Attaching LG', '*C(=O)OCc1ccccc1'), ('Attaching LG', '*O')]"
3,atom_lg_vocab,3,"[('Change Atom', (1, 0)), ('Attaching LG', '*C(=O)OCc1ccccc1'), ('Attaching LG', '*O')]"


## Step 4: 把 edit 序列展开成逐步图状态

### 为什么需要这一步？

Graph2Edits 的训练样本不是“1 条反应 = 1 个样本”，而是“1 条反应 = 多个 time-step 图状态”。

每做完一步 edit，当前产物图都会被改写成新的中间体；模型下一步看到的，就是这个新的图状态。

### 核心思想

- `apply_edit_to_mol()`：把单步 edit 真正作用到分子图上。
- `RxnGraph`：保存某一步的当前产物图、目标 edit、目标原子位置。
- `process_batch_graphs()`：把多条反应的图状态序列压成模型可读的张量。

### 源码对应

- 文件：`source_repos/Graph2Edits/prepare_data.py`
- 函数：`apply_edit_to_mol()`
- 函数：`process_batch()`
- 函数：`prepare_data()`

In [5]:
def normalize_mol(mol):
    """把 edit 后的中间分子重新 round-trip 成可安全取特征的 RDKit Mol。"""
    return Chem.MolFromSmiles(Chem.MolToSmiles(mol))


def expand_reaction_to_graph_seq(rxn_data, use_rxn_class=False):
    reactant_smi, product_smi = rxn_data.rxn_smi.split(">>")
    reactant_mol = Chem.MolFromSmiles(reactant_smi)
    product_mol = Chem.MolFromSmiles(product_smi)
    Chem.Kekulize(product_mol)

    current_mol = normalize_mol(product_mol)
    graph_seq = []
    replay_rows = []
    final_smi = None

    for step_idx, edit in enumerate(rxn_data.edits):
        graph_ready_mol = normalize_mol(current_mol)

        if edit == "Terminate":
            graph = RxnGraph(
                prod_mol=Chem.Mol(graph_ready_mol),
                edit_to_apply=edit,
                reac_mol=Chem.Mol(reactant_mol),
                rxn_class=rxn_data.rxn_class,
                use_rxn_class=use_rxn_class,
            )
            graph_seq.append(graph)
            final_mol = normalize_mol(Termination(action_vocab="Terminate").apply(Chem.Mol(graph_ready_mol)))
            final_smi = Chem.MolToSmiles(final_mol)
            replay_rows.append(
                {
                    "step": step_idx,
                    "edit": "Terminate",
                    "edit_atom": "-",
                    "current_product": Chem.MolToSmiles(graph_ready_mol),
                    "next_state": final_smi,
                }
            )
        else:
            edit_atom = rxn_data.edits_atom[step_idx]
            next_mol = normalize_mol(apply_edit_to_mol(Chem.Mol(graph_ready_mol), edit, edit_atom))
            graph = RxnGraph(
                prod_mol=Chem.Mol(graph_ready_mol),
                edit_to_apply=edit,
                edit_atom=edit_atom,
                reac_mol=Chem.Mol(reactant_mol),
                rxn_class=rxn_data.rxn_class,
                use_rxn_class=use_rxn_class,
            )
            graph_seq.append(graph)
            replay_rows.append(
                {
                    "step": step_idx,
                    "edit": repr(edit),
                    "edit_atom": repr(edit_atom),
                    "current_product": Chem.MolToSmiles(graph_ready_mol),
                    "next_state": Chem.MolToSmiles(next_mol),
                }
            )
            current_mol = Chem.Mol(next_mol)

    return graph_seq, pd.DataFrame(replay_rows), final_smi


def process_batch_graphs(batch_graphs, bond_vocab, atom_vocab, use_rxn_class=False):
    lengths = torch.tensor([len(graph_seq) for graph_seq in batch_graphs], dtype=torch.long)
    max_length = max(lengths.tolist())

    graph_seq_tensors = []
    edit_seq_labels = []
    seq_mask = []

    for step_idx in range(max_length):
        graphs_idx = [
            copy.deepcopy(batch_graphs[i][min(step_idx, length - 1)]).get_components(
                attrs=["prod_graph", "edit_to_apply", "edit_atom"]
            )
            for i, length in enumerate(lengths)
        ]
        mask = (step_idx < lengths).long()
        prod_graphs, edits, edit_atoms = list(zip(*graphs_idx))
        edit_labels = prepare_edit_labels(prod_graphs, edits, edit_atoms, bond_vocab, atom_vocab)
        current_graph_tensors = get_batch_graphs(prod_graphs, use_rxn_class=use_rxn_class)
        graph_seq_tensors.append(current_graph_tensors)
        edit_seq_labels.append(edit_labels)
        seq_mask.append(mask)

    seq_mask = torch.stack(seq_mask).long()
    return graph_seq_tensors, edit_seq_labels, seq_mask


graph_sequences = []
replay_tables = {}
final_reactants = {}

for rxn_data in reaction_objects:
    graph_seq, replay_df, final_smi = expand_reaction_to_graph_seq(rxn_data)
    graph_sequences.append(graph_seq)
    replay_tables[rxn_data.rxn_id] = replay_df
    final_reactants[rxn_data.rxn_id] = final_smi

example_id = demo_df.iloc[1]["id"]
display(replay_tables[example_id])
print("replayed final reactants:", final_reactants[example_id])
print("gold reactants:", reaction_objects[1].rxn_smi.split(">>")[0])

batch_tensors = process_batch_graphs(graph_sequences, bond_vocab, atom_vocab, use_rxn_class=False)
graph_seq_tensors, edit_seq_labels, seq_mask = batch_tensors

shape_rows = []
for step_idx, (graph_tensors, scopes) in enumerate(graph_seq_tensors):
    f_atoms, f_bonds, a2b, b2a, b2revb, undirected_b2a = graph_tensors
    atom_scope, bond_scope = scopes
    shape_rows.append(
        {
            "step": step_idx,
            "f_atoms": tuple(f_atoms.shape),
            "f_bonds": tuple(f_bonds.shape),
            "a2b": tuple(a2b.shape),
            "b2a": tuple(b2a.shape),
            "b2revb": tuple(b2revb.shape),
            "undirected_b2a": tuple(undirected_b2a.shape),
            "atom_scope": atom_scope,
            "bond_scope": bond_scope,
        }
    )

display(pd.DataFrame(shape_rows))
print("seq_mask shape:", tuple(seq_mask.shape))

,step,edit,edit_atom,current_product,next_state
0,0,"('Delete Bond', (None, None))","[2, 3]",[O:1]=[C:2]([NH:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12)[c:14]1[cH:15][c:16]([N+:17](=[O:...,[NH:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2][c:14]1[cH:15][c:16]([N+:17](=[O:1...
1,1,"('Attaching LG', '*O')",2,[NH:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2][c:14]1[cH:15][c:16]([N+:17](=[O:1...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2]([c:14]1[cH:15][c:16]([N+:17](=[O...
2,2,Terminate,-,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2]([c:14]1[cH:15][c:16]([N+:17](=[O...,[NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2]([c:14]1[cH:15][c:16]([N+:17](=[O...


replayed final reactants: [NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2]([c:14]1[cH:15][c:16]([N+:17](=[O:18])[O-:19])[c:20]([S:21][c:22]2[c:23]([Cl:24])[cH:25][n:26][cH:27][c:28]2[Cl:29])[s:30]1)[OH:31]
gold reactants: [NH2:3][c:4]1[cH:5][cH:6][cH:7][c:8]2[cH:9][n:10][cH:11][cH:12][c:13]12.[O:1]=[C:2]([c:14]1[cH:15][c:16]([N+:17](=[O:18])[O-:19])[c:20]([S:21][c:22]2[c:23]([Cl:24])[cH:25][n:26][cH:27][c:28]2[Cl:29])[s:30]1)[OH:31]


,step,f_atoms,f_bonds,a2b,b2a,b2revb,undirected_b2a,atom_scope,bond_scope
0,0,"(85, 85)","(181, 97)","(85, 4)","(181,)","(181,)","(91, 2)","[(1, 27), (28, 30), (58, 27)]","[(1, 27), (28, 33), (61, 30)]"
1,1,"(85, 85)","(179, 97)","(85, 4)","(179,)","(179,)","(90, 2)","[(1, 27), (28, 30), (58, 27)]","[(1, 27), (28, 32), (60, 30)]"
2,2,"(96, 85)","(203, 97)","(96, 4)","(203,)","(203,)","(102, 2)","[(1, 37), (38, 31), (69, 27)]","[(1, 38), (39, 33), (72, 30)]"


seq_mask shape: (3, 3)


## Step 5: 保存教学版产物并映射回原仓库文件流

### 为什么需要这一步？

前面的步骤已经在内存里完成了 Graph2Edits 的预处理主线。这里再把关键中间产物保存到教学目录，形成一个最小可复用数据包，便于下一份 notebook 直接加载。

### 教学版保存策略

- `train.file.kekulized`：保存 `ReactionData` 列表。
- `bond_vocab.txt / atom_vocab.txt / lg_vocab.txt / atom_lg_vocab.txt`：保存词表。
- `without_rxn_class/batch-0.pt`：保存 3 条反应压成的最小 batch。

### 与原仓库的差异

原仓库把这些文件写到 `source_repos/Graph2Edits/data/...`；教学版为了避免污染源码目录，统一写到教程目录下的 `data/processed/uspto_50k_demo/train/`。

In [6]:
PROCESSED_TRAIN_ROOT.mkdir(parents=True, exist_ok=True)
TENSOR_ROOT.mkdir(parents=True, exist_ok=True)

joblib.dump(reaction_objects, PROCESSED_TRAIN_ROOT / "train.file.kekulized", compress=3)
joblib.dump(atom_edits, PROCESSED_TRAIN_ROOT / "atom_vocab.txt")
joblib.dump(bond_edits, PROCESSED_TRAIN_ROOT / "bond_vocab.txt")
joblib.dump(lg_edits, PROCESSED_TRAIN_ROOT / "lg_vocab.txt")
joblib.dump(atom_lg_edits, PROCESSED_TRAIN_ROOT / "atom_lg_vocab.txt")
torch.save(batch_tensors, TENSOR_ROOT / "batch-0.pt")

print("saved files:")
for path in [
    PROCESSED_TRAIN_ROOT / "train.file.kekulized",
    PROCESSED_TRAIN_ROOT / "atom_vocab.txt",
    PROCESSED_TRAIN_ROOT / "bond_vocab.txt",
    PROCESSED_TRAIN_ROOT / "lg_vocab.txt",
    PROCESSED_TRAIN_ROOT / "atom_lg_vocab.txt",
    TENSOR_ROOT / "batch-0.pt",
]:
    print("-", path.relative_to(PROJECT_ROOT))

train_dataset = RetroEditDataset(str(TENSOR_ROOT))
eval_dataset = RetroEvalDataset(str(PROCESSED_TRAIN_ROOT), "train.file.kekulized", use_rxn_class=False)
loaded_batch = torch_load_compat(TENSOR_ROOT / "batch-0.pt")

print("\ntrain_dataset batches:", len(train_dataset))
print("eval_dataset reactions:", len(eval_dataset))
print("loaded batch object types:", [type(x).__name__ for x in loaded_batch])

saved files:
- teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/train.file.kekulized
- teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/atom_vocab.txt
- teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/bond_vocab.txt
- teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/lg_vocab.txt
- teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/atom_lg_vocab.txt
- teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/processed/uspto_50k_demo/train/without_rxn_class/batch-0.pt

train_dataset batches: 1
eval_dataset reactions: 3
loaded batch object types: ['list', 'list', 'Tensor']


## 小结

这份 notebook 完成了 Graph2Edits 数据处理的最小闭环：

1. 从 raw reaction 回到 canonicalized reaction。
2. 从 canonicalized reaction 提取 edit 序列与 `edits_atom`。
3. 从 edit 序列构造 vocab。
4. 从单条反应展开出多步图状态序列。
5. 把多条反应压成模型可读取的 batch 张量并写入教学目录。

下一份 notebook `3_模型展示.ipynb` 会在这些教学版产物之上，解释 `MolGraph -> MPNEncoder -> edit logits -> autoregressive editing / beam search` 的模型主线。